# Senator Feature Base Functions

In [ ]:
# Senatör Featureunda yapmak istediğim:
# Senatörlerin rankingini sağlamak bu 2 farklı yolun birleşmesiyle olucak
# Biri senator trust index diyebiliriz bu geçmişe yönelik işlemlerde
# Bunda 2 spektrum yaparız buy farklı sell farklı olur 
# Transaction date ve received datedeki fiyatları karşılaştıracak
# Bu sayede aradaki sürede fiyat farkı az mı olmuş uzun vadeli mi
# Kısa vadeli mi yatırım yapıyor görüyor olacağız
# İkincisi senatör performans endeksi backtestle beraber
# Bir nevi senatörlerin performanslarını yorumlayacağız
# Bu elde ettiğimiz rankingi kullanacğımız bir faz olucak
# Ekonomik metriklerle sıraladığımız ve belli bir değerin üstünü yatırım
# için değerlendirmeye geçmeden yakın zamanda senatörler kendi
# rating katsayılarıyla beraber buy pozitif etki edicek
# sell negatif etki edicek sonrasında yine ranking hisselerimiz olacak

# Ayrıca yapılacaklar: 
# Otomatik power bi raporları
# JupyterHUBA geçiş ya da benzeri
# Trade box enjeksityonu
# Zipline a bak https://zipline.ml4trading.io



In [41]:
import pandas as pd
import numpy as np
import time, json, ssl, certifi
from datetime import datetime, timedelta
from urllib.request import urlopen

API_KEY = open("FMP API KEY.txt").read().strip()
BASE_URL = "https://financialmodelingprep.com/api/v3/"

# Excel dosyasından S&P500 şirketleri
df = pd.read_excel("listof90year.xlsx")
tickers = df.drop(columns="year").values.flatten()
tickers = pd.Series(tickers).dropna().unique().tolist()

def get_jsonparsed_data(url: str):
    context = ssl.create_default_context(cafile=certifi.where())
    with urlopen(url, context=context) as resp:
        return json.loads(resp.read().decode("utf-8"))

def fmp_close_series(ticker: str, start: str, end: str) -> pd.Series:
    url = f"{BASE_URL}historical-price-full/{ticker}?from={start}&to={end}&apikey={API_KEY}"
    js = get_jsonparsed_data(url)
    if not js or "historical" not in js or not js["historical"]:
        return pd.Series()
    df = pd.DataFrame(js["historical"])
    df["date"] = pd.to_datetime(df["date"])
    df = df.set_index("date").sort_index()
    return df["close"]

# Senatör puanı: Alım - Fiyat artışı ilişkisi
def collect_senator_trust_scores(tickers, start_range="2021-01-01", end_range="2024-01-01", limit=50):
    senator_scores = {}

    for i, ticker in enumerate(tickers[:limit]):
        try:
            url = f"https://financialmodelingprep.com/api/v4/senate-trading?symbol={ticker}&apikey={API_KEY}"
            data = get_jsonparsed_data(url)
            trades_df = pd.DataFrame(data)

            if trades_df.empty:
                continue

            trades_df['transactionDate'] = pd.to_datetime(trades_df['transactionDate'])
            trades_df['dateRecieved'] = pd.to_datetime(trades_df['dateRecieved'])

            trades_df = trades_df[trades_df['type'].str.lower().isin(['purchase', 'buy'])]
            mask = (trades_df['transactionDate'] >= pd.to_datetime(start_range)) & \
                   (trades_df['transactionDate'] <= pd.to_datetime(end_range))
            trades_df = trades_df.loc[mask]
            trades_df['senator'] = trades_df['firstName'] + ' ' + trades_df['lastName']

            for _, row in trades_df.iterrows():
                s_name = row['senator']
                td = row['transactionDate']
                rd = row['dateRecieved']

                # Fiyat verisi
                trans_prices = fmp_close_series(ticker, (td - timedelta(days=3)).strftime('%Y-%m-%d'),
                                                     (td + timedelta(days=3)).strftime('%Y-%m-%d'))
                recv_prices  = fmp_close_series(ticker, (rd - timedelta(days=3)).strftime('%Y-%m-%d'),
                                                     (rd + timedelta(days=3)).strftime('%Y-%m-%d'))

                if trans_prices.empty or recv_prices.empty:
                    continue

                try:
                    transaction_price = trans_prices.loc[trans_prices.index.get_loc(td, method='nearest')]
                    received_price = recv_prices.loc[recv_prices.index.get_loc(rd, method='nearest')]
                except:
                    continue

                ret = (received_price - transaction_price) / transaction_price
                senator_scores.setdefault(s_name, []).append(ret)

                time.sleep(0.1)

        except Exception as e:
            print(f"Error on {ticker}: {e}")
            continue

    # Ortalama skorları hesapla
    final_scores = []
    for name, returns in senator_scores.items():
        if len(returns) >= 3:  # En az 3 işlem yapılmış olmalı
            avg_score = np.mean(returns)
            final_scores.append({
                "Senator": name,
                "Avg Trust Score (%)": round(avg_score * 100, 2),
                "Trade Count": len(returns)
            })

    rank_df = pd.DataFrame(final_scores)
    rank_df = rank_df.sort_values("Avg Trust Score (%)", ascending=False).reset_index(drop=True)
    return rank_df

# Çalıştırmak için (örnek: ilk 50 S&P 500 şirketi)
df_scores = collect_senator_trust_scores(tickers, "2021-01-01", "2024-01-01", limit=50)
print(df_scores.head(10))

KeyError: 'Avg Trust Score (%)'

In [1]:
# Senator verisi çekme
import requests
import pandas as pd
from datetime import datetime, timedelta

symbol = "AAPL"
API_KEY = open("FMP API KEY.txt").read().strip()
url = f"https://financialmodelingprep.com/api/v4/senate-trading?symbol={symbol}&apikey={API_KEY}"

response = requests.get(url)
data = response.json()

df = pd.DataFrame(data)
df['transactionDate'] = pd.to_datetime(df['transactionDate'])
df.sort_values(by='transactionDate', ascending=False, inplace=True)
df

,firstName,lastName,office,link,dateRecieved,transactionDate,owner,assetDescription,assetType,type,amount,comment,symbol
0,Tommy,Tuberville,Tommy Tuberville,https://efdsearch.senate.gov/search/view/ptr/6...,2025-05-15,2025-04-15,Joint,Apple Inc,Stock,Sale,"$15,001 - $50,000",,AAPL
1,Tommy,Tuberville,Tommy Tuberville,https://efdsearch.senate.gov/search/view/ptr/6...,2025-05-14,2025-04-15,Joint,Apple Inc,Stock,Sale (Full),"$15,001 - $50,000",--,AAPL
2,Sheldon,Whitehouse,Sheldon Whitehouse,https://efdsearch.senate.gov/search/view/ptr/3...,2025-05-13,2025-04-14,Self,Apple Inc,Stock,Sale,"$15,001 - $50,000",,AAPL
3,Sheldon,Whitehouse,Sheldon Whitehouse,https://efdsearch.senate.gov/search/view/ptr/3...,2025-05-12,2025-04-14,Self,Apple Inc,Stock,Sale (Partial),"$15,001 - $50,000",--,AAPL
4,Shelley,Moore Capito,Shelley Moore Capito,https://efdsearch.senate.gov/search/view/ptr/2...,2025-03-05,2025-02-05,Spouse,Apple Inc,Stock,Sale (Partial),"$1,001 - $15,000",--,AAPL
...,...,...,...,...,...,...,...,...,...,...,...,...,...
261,Sheldon,Whitehouse,Sheldon Whitehouse,https://efdsearch.senate.gov/search/view/ptr/7...,2015-08-12,2014-03-13,Joint,Apple Inc. (NASDAQ),,Sale (Partial),"$1,001 - $15,000",--,AAPL
288,Pat,Roberts,Pat Roberts,https://efdsearch.senate.gov/search/view/ptr/f...,2014-04-03,2014-03-07,Spouse,Apple Inc. (NASDAQ),,Purchase,"$50,001 - $100,000",--,AAPL
287,Pat,Roberts,Pat Roberts,https://efdsearch.senate.gov/search/view/ptr/f...,2014-04-03,2014-03-03,Spouse,Apple Inc. (NASDAQ),,Sale (Full),"$50,001 - $100,000",--,AAPL
291,Sheldon,Whitehouse,Sheldon Whitehouse,https://efdsearch.senate.gov/search/view/ptr/1...,2014-02-27,2014-02-10,Joint,Apple Inc. (NASDAQ),,Sale (Partial),"$15,001 - $50,000",--,AAPL


In [ ]:
# Güncel S&P 500'de kimler olduğunu gösteriyor
import time, json, certifi
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import ssl
from urllib.request import urlopen
def get_jsonparsed_data(url: str):
    """
    URL → JSON → Python objesi.
    Modern SSL context kullanır, DeprecationWarning engellenir.
    """
    context = ssl.create_default_context(cafile=certifi.where())
    with urlopen(url, context=context) as resp:
        return json.loads(resp.read().decode("utf-8"))
def get_sp500_constituents() -> pd.DataFrame:
    url = f"https://financialmodelingprep.com/api/v3/sp500_constituent?apikey={API_KEY}"
    data = get_jsonparsed_data(url)
    df = pd.DataFrame(data)
    return df[['symbol', 'name', 'sector', 'subSector']]
sp500_df = get_sp500_constituents()
print(sp500_df.head())

  symbol                name                  sector  \
0   COIN     Coinbase Global      Financial Services   
1   DASH            DoorDash  Communication Services   
2    EXE       Expand Energy                  Energy   
3    TKO  TKO Group Holdings  Communication Services   
4    WSM     Williams-Sonoma       Consumer Cyclical   

                            subSector  
0  Financial - Data & Stock Exchanges  
1      Internet Content & Information  
2  Oil & Gas Exploration & Production  
3                       Entertainment  
4                    Specialty Retail  


In [7]:
sp500_symbols = sp500_df['symbol'].dropna().unique().tolist()


In [39]:
sp500_df

,symbol,name,sector,subSector
0,COIN,Coinbase Global,Financial Services,Financial - Data & Stock Exchanges
1,DASH,DoorDash,Communication Services,Internet Content & Information
2,EXE,Expand Energy,Energy,Oil & Gas Exploration & Production
3,TKO,TKO Group Holdings,Communication Services,Entertainment
4,WSM,Williams-Sonoma,Consumer Cyclical,Specialty Retail
...,...,...,...,...
498,SO,Southern Company,Utilities,Regulated Electric
499,SPGI,S&P Global,Financial Services,Financial - Data & Stock Exchanges
500,UNP,Union Pacific Corporation,Industrials,Railroads
501,XEL,Xcel Energy,Utilities,Regulated Electric


In [28]:
df = pd.read_excel("listof90year.xlsx")
long_df = df.melt(id_vars="year", value_name="ticker")[["year", "ticker"]]
long_df = long_df.dropna()
long_df["presence"] = 1
pivot_df = long_df.pivot_table(index="ticker", columns="year", values="presence", fill_value=0)
pivot_df = pivot_df.sort_index(axis=1)
pivot_df = pivot_df.reset_index()

In [ ]:
pivot_df.loc[pivot_df["ticker"] == "AAPL", 2009].values[0] == 1

True